# 05 Test results
Latest run per model, by training audio (raw vs trimmed). EER, minDCF and F1 recomputed from each run's score
files. F1 uses the dev EER threshold with spoof as the positive class; minDCF uses the project costs (miss 1,
false alarm 10, spoof prior 0.05).

In [1]:
import pandas as pd
from audio_df.helpers.research.runs import collect, eer, metrics_by_testset

MODELS = ["a1", "a4", "a2", "d1", "d2", "d3"]
SETS = ["dev", "eval", "eval_mp3_32k", "eval_noise_10db", "eval_opus_16k", "eval_resample_8k", "itw",
        "asvspoof2021_eval", "asvspoof2021_hidden", "asvspoof2021df_eval", "asvspoof2021df_hidden"]
results = collect("../runs", MODELS)

per_model = {k: metrics_by_testset(v["scores"], SETS) for k, v in results.items()}

def metric_table(name):  # mean over the 10 test sets, only when a model has all of them
    t = pd.DataFrame({k: m[name] for k, m in per_model.items()}).T.reindex(columns=SETS).sort_index()
    t.index.names = ["model", "train"]
    t["mean"] = t[SETS[1:]].mean(axis=1, skipna=False)
    return t

pd.DataFrame({k: {"artifact": v["artifact"], "runs": ", ".join(v["runs"])} for k, v in results.items()}).T

,,artifact,runs
a2,raw,20261020-043749-946513,"E-20261020-043749-966034-a2, E-20261020-174446..."
d1,raw,20261020-053911-536394,"E-20261020-053911-580516-d1, E-20261020-195806..."
d2 L12,raw,20261020-080344-358634,"E-20261020-080344-411189-d2, E-20261020-200216..."
a1,raw,20261020-134519-735144,"E-20261020-134620-502536-a1, E-20261020-153158..."
a4,raw,20261020-193844-844933,E-20261020-194017-076363-a4
d3 L12,raw,20261021-013650-100255,E-20261021-013825-460640-d3
a1,trimmed,20261023-171757-192173,E-20261023-172139-766981-a1
a4,trimmed,20261023-194455-167994,E-20261023-194755-458184-a4
a2,trimmed,20261023-200629-150334,E-20261023-200932-976565-a2
d1,trimmed,20261023-201429-262937,E-20261023-201728-785995-d1


## EER % per test set
D2/D3 are labelled with their WavLM layer (L12 = round 2, L6 = round 3). A1/A4 DF results are on 10,000-clip
samples (notebook 01).

In [2]:
metric_table("EER %").round(2)

dev   eval  eval_mp3_32k  eval_noise_10db  eval_opus_16k  \
model  train                                                                 
a1     raw      12.35  21.05         42.54            41.20          25.56   
       trimmed  12.28  20.49         39.25            41.57          25.27   
a2     raw       5.49  11.95         13.65            11.63          13.48   
       trimmed   1.37   7.64          8.42            16.95           7.96   
a4     raw      38.50  36.07         46.71            31.78          39.82   
       trimmed  40.46  37.04         47.39            36.22          38.95   
d1     raw       7.38  12.31         35.76            13.76          16.38   
       trimmed   1.49   7.79         21.91            17.92           9.02   
d2 L12 raw      28.02  28.85         39.06            26.74          36.72   
       trimmed   6.99  11.77         12.98            27.80          11.94   
d2 L6  trimmed   1.72   7.19          8.45            13.76           7.66   
d3 L12 raw      31.91  22.69         54.38            28.80          52.80   
       trimmed   5.38  12.63         17.35            26.26          11.63   
d3 L6  trimmed   2.35   6.55          9.19            12.02           7.35   

                eval_resample_8k    itw  asvspoof2021_eval  \
model  train                                                 
a1     raw                 35.47  56.98                NaN   
       trimmed             36.42  58.25              34.08   
a2     raw                 12.11  15.74                NaN   
       trimmed              7.45  19.17              14.28   
a4     raw                 39.83  74.23                NaN   
       trimmed             39.34  71.99              40.59   
d1     raw                 14.63  39.23                NaN   
       trimmed              9.97  33.21              16.80   
d2 L12 raw                 30.42  14.97                NaN   
       trimmed             12.31  21.31              19.46   
d2 L6  trimmed              7.48  19.93              10.09   
d3 L12 raw                 23.74  45.06                NaN   
       trimmed             12.02  22.51              20.36   
d3 L6  trimmed              6.59  22.68              10.09   

                asvspoof2021_hidden  asvspoof2021df_eval  \
model  train                                               
a1     raw                      NaN                  NaN   
       trimmed                42.86                29.12   
a2     raw                      NaN                  NaN   
       trimmed                20.92                10.74   
a4     raw                      NaN                  NaN   
       trimmed                44.29                35.25   
d1     raw                      NaN                  NaN   
       trimmed                21.89                22.62   
d2 L12 raw                      NaN                  NaN   
       trimmed                20.56                  NaN   
d2 L6  trimmed                14.64                10.02   
d3 L12 raw                      NaN                  NaN   
       trimmed                21.52                  NaN   
d3 L6  trimmed                16.08                11.18   

                asvspoof2021df_hidden   mean  
model  train                                  
a1     raw                        NaN    NaN  
       trimmed                  32.12  35.94  
a2     raw                        NaN    NaN  
       trimmed                  11.66  12.52  
a4     raw                        NaN    NaN  
       trimmed                  41.88  43.29  
d1     raw                        NaN    NaN  
       trimmed                  29.33  19.05  
d2 L12 raw                        NaN    NaN  
       trimmed                    NaN    NaN  
d2 L6  trimmed                  10.71  10.99  
d3 L12 raw                        NaN    NaN  
       trimmed                    NaN    NaN  
d3 L6  trimmed                  13.53  11.53

## minDCF per test set
Lower is better; 1.0 is no better than always answering the cheaper class.

In [3]:
metric_table("minDCF").round(3)

dev   eval  eval_mp3_32k  eval_noise_10db  eval_opus_16k  \
model  train                                                                 
a1     raw      0.217  0.476         0.987            0.917          0.596   
       trimmed  0.211  0.470         0.937            0.924          0.590   
a2     raw      0.137  0.294         0.330            0.310          0.319   
       trimmed  0.033  0.158         0.176            0.429          0.181   
a4     raw      0.969  0.946         0.996            0.799          0.983   
       trimmed  0.956  0.953         0.997            0.856          0.982   
d1     raw      0.207  0.322         0.877            0.363          0.450   
       trimmed  0.039  0.186         0.578            0.450          0.227   
d2 L12 raw      0.670  0.761         0.929            0.772          0.907   
       trimmed  0.173  0.300         0.306            0.685          0.295   
d2 L6  trimmed  0.044  0.168         0.192            0.335          0.172   
d3 L12 raw      0.843  0.657         1.000            0.782          1.000   
       trimmed  0.144  0.310         0.475            0.628          0.268   
d3 L6  trimmed  0.061  0.156         0.203            0.307          0.178   

                eval_resample_8k    itw  asvspoof2021_eval  \
model  train                                                 
a1     raw                 0.796  0.891                NaN   
       trimmed             0.774  0.972              0.709   
a2     raw                 0.288  0.417                NaN   
       trimmed             0.151  0.515              0.346   
a4     raw                 0.932  1.000                NaN   
       trimmed             0.875  1.000              0.867   
d1     raw                 0.349  0.999                NaN   
       trimmed             0.216  0.962              0.426   
d2 L12 raw                 0.797  0.371                NaN   
       trimmed             0.289  0.579              0.550   
d2 L6  trimmed             0.158  0.476              0.248   
d3 L12 raw                 0.598  0.999                NaN   
       trimmed             0.282  0.625              0.561   
d3 L6  trimmed             0.145  0.577              0.261   

                asvspoof2021_hidden  asvspoof2021df_eval  \
model  train                                               
a1     raw                      NaN                  NaN   
       trimmed                0.911                0.834   
a2     raw                      NaN                  NaN   
       trimmed                0.534                0.260   
a4     raw                      NaN                  NaN   
       trimmed                0.998                0.879   
d1     raw                      NaN                  NaN   
       trimmed                0.547                0.594   
d2 L12 raw                      NaN                  NaN   
       trimmed                0.518                  NaN   
d2 L6  trimmed                0.387                0.244   
d3 L12 raw                      NaN                  NaN   
       trimmed                0.555                  NaN   
d3 L6  trimmed                0.419                0.282   

                asvspoof2021df_hidden   mean  
model  train                                  
a1     raw                        NaN    NaN  
       trimmed                  0.821  0.794  
a2     raw                        NaN    NaN  
       trimmed                  0.275  0.303  
a4     raw                        NaN    NaN  
       trimmed                  0.997  0.941  
d1     raw                        NaN    NaN  
       trimmed                  0.795  0.498  
d2 L12 raw                        NaN    NaN  
       trimmed                    NaN    NaN  
d2 L6  trimmed                  0.258  0.264  
d3 L12 raw                        NaN    NaN  
       trimmed                    NaN    NaN  
d3 L6  trimmed                  0.314  0.284

## F1 per test set
At the dev EER threshold, spoof = positive, higher is better. Spoof is about 90% of most sets, so F1 stays high
even for weak models; read it next to EER and minDCF.

In [4]:
metric_table("F1 (spoof)").round(3)

dev   eval  eval_mp3_32k  eval_noise_10db  eval_opus_16k  \
model  train                                                                 
a1     raw      0.927  0.798         0.925            0.743          0.934   
       trimmed  0.928  0.809         0.943            0.811          0.942   
a2     raw      0.969  0.852         0.895            0.764          0.938   
       trimmed  0.992  0.909         0.945            0.938          0.964   
a4     raw      0.741  0.605         0.939            0.733          0.721   
       trimmed  0.725  0.472         0.888            0.600          0.613   
d1     raw      0.958  0.808         0.946            0.741          0.956   
       trimmed  0.992  0.892         0.946            0.895          0.968   
d2 L12 raw      0.822  0.832         0.437            0.335          0.820   
       trimmed  0.960  0.867         0.954            0.929          0.953   
d2 L6  trimmed  0.990  0.908         0.967            0.948          0.967   
d3 L12 raw      0.793  0.794         0.152            0.505          0.826   
       trimmed  0.969  0.844         0.950            0.935          0.957   
d3 L6  trimmed  0.987  0.907         0.964            0.956          0.971   

                eval_resample_8k    itw  asvspoof2021_eval  \
model  train                                                 
a1     raw                 0.939  0.361                NaN   
       trimmed             0.945  0.382              0.925   
a2     raw                 0.934  0.792                NaN   
       trimmed             0.965  0.719              0.936   
a4     raw                 0.229  0.040                NaN   
       trimmed             0.003  0.038              0.221   
d1     raw                 0.424  0.505                NaN   
       trimmed             0.958  0.589              0.932   
d2 L12 raw                 0.854  0.814                NaN   
       trimmed             0.901  0.559              0.914   
d2 L6  trimmed             0.930  0.633              0.944   
d3 L12 raw                 0.682  0.461                NaN   
       trimmed             0.887  0.551              0.919   
d3 L6  trimmed             0.936  0.640              0.941   

                asvspoof2021_hidden  asvspoof2021df_eval  \
model  train                                               
a1     raw                      NaN                  NaN   
       trimmed                0.898                0.908   
a2     raw                      NaN                  NaN   
       trimmed                0.944                0.922   
a4     raw                      NaN                  NaN   
       trimmed                0.390                0.475   
d1     raw                      NaN                  NaN   
       trimmed                0.933                0.942   
d2 L12 raw                      NaN                  NaN   
       trimmed                0.929                  NaN   
d2 L6  trimmed                0.950                0.968   
d3 L12 raw                      NaN                  NaN   
       trimmed                0.932                  NaN   
d3 L6  trimmed                0.949                0.958   

                asvspoof2021df_hidden   mean  
model  train                                  
a1     raw                        NaN    NaN  
       trimmed                  0.828  0.839  
a2     raw                        NaN    NaN  
       trimmed                  0.937  0.918  
a4     raw                        NaN    NaN  
       trimmed                  0.697  0.440  
d1     raw                        NaN    NaN  
       trimmed                  0.929  0.898  
d2 L12 raw                        NaN    NaN  
       trimmed                    NaN    NaN  
d2 L6  trimmed                  0.954  0.917  
d3 L12 raw                        NaN    NaN  
       trimmed                    NaN    NaN  
d3 L6  trimmed                  0.950  0.917

## D2 vs D3 (round 3, WavLM layer 6)

In [5]:
pairs = {m: metric_table(m).xs("trimmed", level="train").loc[["d2 L6", "d3 L6"]] for m in ["EER %", "minDCF"]}
cmp = pd.DataFrame({f"{m} {k}": pairs[m].loc[f"{k} L6"] for m in pairs for k in ["d2", "d3"]})
cmp["better (EER)"] = (cmp["EER % d2"] < cmp["EER % d3"]).map({True: "D2", False: "D3"})
cmp.round(3)

,EER % d2,EER % d3,minDCF d2,minDCF d3,better (EER)
dev,1.720,2.355,0.044,0.061,D2
eval,7.193,6.554,0.168,0.156,D3
eval_mp3_32k,8.447,9.191,0.192,0.203,D2
eval_noise_10db,13.760,12.018,0.335,0.307,D3
eval_opus_16k,7.658,7.351,0.172,0.178,D3
eval_resample_8k,7.477,6.590,0.158,0.145,D3
itw,19.931,22.679,0.476,0.577,D2
asvspoof2021_eval,10.091,10.092,0.248,0.261,D2
asvspoof2021_hidden,14.641,16.077,0.387,0.419,D2
asvspoof2021df_eval,10.021,11.184,0.244,0.282,D2


D2 wins 7 of 11 sets and the mean (EER 11.0% vs 11.5%, minDCF 0.264 vs 0.284). D3 only wins on audio derived from the 2019 corpus (2019 eval, MUSAN noise, Opus, 8 kHz); D2 wins every cross-dataset set (In-the-Wild 19.9% vs 22.7%, 2021 LA hidden, 2021 DF eval and hidden). D3 fits the training corpus slightly better; D2 generalises better. Both from one training run each.

## Per attack, 2019 eval (EER %)

In [6]:
def per_attack(df):
    bona = df[df.label == "bonafide"]
    return {a: eer(pd.concat([bona, g])) for a, g in df[df.label == "spoof"].groupby("attack_id")}

pd.DataFrame({k: per_attack(v["scores"]["eval"]) for k, v in results.items()}).T.round(1)

,,A07,A08,A09,A10,A11,A12,A13,A14,A15,A16,A17,A18,A19
a2,raw,11.9,4.9,1.5,35.3,9.5,10.1,0.7,14.3,14.5,8.8,2.1,5.8,8.9
d1,raw,9.0,6.5,2.9,26.8,10.2,13.9,1.0,17.7,19.0,8.4,4.4,3.5,6.0
d2 L12,raw,28.8,12.9,7.9,37.5,20.9,19.4,29.9,33.5,32.0,27.7,27.9,34.1,44.3
a1,raw,29.0,1.3,0.3,37.1,1.3,37.4,27.1,4.4,6.0,33.2,2.7,22.6,12.0
a4,raw,35.3,36.7,25.5,38.9,39.0,42.8,17.3,34.1,38.1,38.9,43.0,41.8,32.7
d3 L12,raw,17.9,21.4,17.3,31.6,22.9,22.0,19.8,37.4,35.5,19.7,17.4,18.5,22.5
a1,trimmed,28.9,1.1,0.2,38.1,1.5,37.7,26.6,5.1,6.5,31.4,2.8,19.8,8.7
a4,trimmed,37.3,36.8,22.1,40.5,38.9,44.9,30.5,35.3,30.3,40.3,38.3,43.5,39.5
a2,trimmed,6.6,2.0,0.5,30.9,7.5,6.2,0.2,1.6,11.0,3.1,0.1,0.6,2.2
d1,trimmed,5.8,2.4,1.5,23.6,5.4,9.3,0.3,3.1,15.3,3.8,0.7,0.8,2.3


## Conclusion
- Trimmed training fixed the silence shortcut for the WavLM models; the LFCC-only models (A1, A4) stay weak on every metric (A4's minDCF is close to 1).
- Round 3 D2/D3 (layer 6) beat A2 on mean EER and minDCF over the 10 test sets (minDCF D2 0.264, D3 0.284, A2 0.303); F1 is about the same for all three (≈0.92) because spoof dominates every set.
- D2 is the best general model, D3 the best on 2019-like audio. In-the-Wild stays around 20% for every model (notebooks 04, 07).

## Arbitration test case
Clips for `uv run arbitration` (`CLIPS` in `pipelines/arbitration.py`, used in notebook 06). These are
In-the-Wild clips where the six detectors (A1, A2, A4, D1, D2 and D3; trimmed training, D2/D3 with WavLM
layer 6) do not all agree at their dev EER thresholds: `11249.wav` plus 200 sampled with a fixed seed.
Numbers are file names in `release_in_the_wild/`.

In [2]:
from audio_df.helpers.research.arbitration_notebook import build_cases

print(build_cases(results, models=("a1", "a2", "a4", "d1", "d2 L6", "d3 L6")))

detectors split on 29,023 of 31,779 In-the-Wild clips
[11249, 5707, 3589, 7423, 8056, 12496, 24621, 23201, 3313, 7082, 30199, 28559, 28997, 24805, 7216, 7435, 14433, 30336, 3610, 27864, 12291, 30593, 9440, 21797, 1421, 544, 14930, 18847, 29905, 18251, 27237, 10337, 12063, 2647, 12326, 9102, 8475, 31727, 22447, 6881, 29553, 14497, 28378, 26262, 8598, 7469, 30489, 11126, 11507, 10197, 9617, 25472, 27730, 10117, 13760, 2574, 1610, 22491, 27584, 2026, 12732, 27112, 31543, 13892, 26388, 3710, 14328, 24773, 2781, 6081, 25822, 25652, 6768, 26035, 16806, 908, 12746, 26366, 3626, 6766, 22196, 2251, 8177, 939, 13745, 20045, 8561, 9332, 17177, 3227, 15959, 18976, 11010, 12188, 1380, 20338, 20794, 11783, 17292, 2106, 18700, 17199, 13909, 16292, 5463, 12515, 29974, 28982, 26102, 23584, 18003, 11543, 15918, 25206, 19516, 1520, 23167, 29388, 21810, 7105, 6116, 13544, 12730, 26260, 24583, 26234, 22373, 4342, 22824, 4838, 6386, 14087, 22645, 4602, 31616, 1909, 5236, 21864, 11621, 3664, 12816, 9978, 311